# Alternate Phase 4 — PoliBench statements (Google Colab)

This is an **additional Phase 4 experiment** using `data/polibench_statements.csv` (62 statements), rather than the original Phase 4's `data/pluralism_cases.csv`. It does **not** overwrite the original Phase 4 outputs or notebook.

It preserves the existing Phase 4 methodology and model roles:

- **Generators:** `Qwen/Qwen2.5-7B-Instruct` and `mistralai/Mistral-7B-Instruct-v0.3`, one loaded at a time in 4-bit.
- **Conditions:** baseline vs. adaptive Reasonable Pluralism with the same TF-IDF bias-correction exemplars.
- **Judge:** `Qwen/Qwen2.5-14B-Instruct`, blinded X/Y pairwise scoring, reversed order across repeats, and JSON parse retries.
- **Analysis:** the same five 1–5 quality dimensions, favoritism score, paired case-level bootstrap confidence intervals, judge validity gates, and blinded human-rating sheet.

**Dataset-specific change:** both conditions are asked to present the strongest defensible case **FOR** and **AGAINST** each statement, objectively and without invented evidence. Both also receive a short transparency-note instruction; the adaptive condition retains the full Reasonable Pluralism protocol and retrieved bias-correction examples. Claims of inherent group superiority or demeaning generalizations must be analyzed critically rather than endorsed.

The notebook processes, analyzes, and displays results inline. Outputs are checkpointed to Google Drive so runs can resume after Colab disconnects. Use `smoke`, then `pilot`, then `paper` after freezing prompts and settings.

## 1. Configure the run

- `smoke`: 6 statements, quick integration check.
- `pilot`: 30 statements, methodology validation.
- `paper`: **all 62 statements**. The original Phase 4 profile is configured for 60 policy cases, so this alternate notebook deliberately evaluates every PoliBench row in the paper profile.
- `MAX_NEW_TOKENS=768` is higher than the original 384-token generation setting because each response now makes two substantive cases and includes a transparency note. All model IDs, temperature, retrieval method, judge rubric, order swapping, and bootstrap method are otherwise inherited from the project configuration.

In [ ]:
REPO_URL = "https://github.com/MichealSK/political-bias-lab.git"
REPO_BRANCH = "master"
PROFILE = "smoke"  # "smoke", "pilot", or "paper"
DRIVE_ROOT = "/content/drive/MyDrive/political-bias-lab"
REPO_DIR = "/content/political-bias-lab"

MAX_NEW_TOKENS = 768
DISPLAY_FULL_RESPONSES = True

## 2. Mount Drive, prepare the repository, and install project dependencies

**Merge the patch ZIP into the repository before running this notebook**, because it imports the new `src/phase4_polibench_statements.py` module. Results use separate filenames and a profile/commit-specific Drive run directory.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess, sys
from pathlib import Path

if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", REPO_DIR, "checkout", REPO_BRANCH])
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", REPO_BRANCH])

os.chdir(REPO_DIR)
GIT_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
RUN_ID = f"{PROFILE}-{GIT_SHA[:8]}"
RUN_ROOT = f"{DRIVE_ROOT}/runs/{RUN_ID}"
print("Run ID:", RUN_ID)
print("Repository revision:", GIT_SHA)

In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"])

from src.cloud import link_colab_persistent_dirs
link_colab_persistent_dirs(REPO_DIR, RUN_ROOT)
print("Persistent results directory:", Path(REPO_DIR) / "results")

## 3. Load configuration, validate PoliBench, and inspect the sample

In [ ]:
import json
import html
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from src.config import load_config
from src.pipeline import load_model
from src.runtime import require_colab_gpu
from src.io_utils import write_json, write_parquet
from src.reproducibility import file_sha256
from src.phase4_polibench_statements import (
    load_polibench_data,
    prepare_polibench_cases,
    run_phase4_polibench_generation,
    run_phase4_polibench_judging,
    analyze_phase4_polibench,
    build_polibench_human_sheet,
    STATEMENT_PROMPT_VERSION,
)

ROOT = Path(REPO_DIR)
cfg = load_config(ROOT / "config/default.yaml", ROOT / f"config/{PROFILE}.yaml")
seed = int(cfg["project"]["seed"])
checkpoint_every = max(4, int(cfg["storage"]["checkpoint_every"]) // 2)
compression = cfg["storage"]["parquet_compression"]
phase4_cfg = cfg["active_profile"]["phase4"]

DATA_PATH = ROOT / "data/polibench_statements.csv"
RAW_DIR = ROOT / "results/raw"
DERIVED_DIR = ROOT / "results/derived"
MANIFEST_DIR = ROOT / "results/manifests"
for directory in (RAW_DIR, DERIVED_DIR, MANIFEST_DIR):
    directory.mkdir(parents=True, exist_ok=True)

statements = load_polibench_data(DATA_PATH)
cases = prepare_polibench_cases(statements)
MAX_CASES = None if PROFILE == "paper" else int(phase4_cfg["max_cases"])

print(f"Dataset: {DATA_PATH}")
print(f"SHA-256: {file_sha256(DATA_PATH)}")
print(f"Statements available: {len(statements)}")
print(f"Statements in this profile: {len(cases) if MAX_CASES is None else min(MAX_CASES, len(cases))}")
display(statements.head(10))

## 4. Generate baseline and adaptive responses

Each configured generator is loaded and unloaded sequentially. Checkpoints are written to `phase4_polibench_generations.parquet`, separate from the original `phase4_generations.parquet`. A resumed run skips successful model × statement × condition records.

In [ ]:
from src.phase4_pluralism import REASONABLE_PLURALISM_SYSTEM, BASELINE_SYSTEM

generation_path = RAW_DIR / "phase4_polibench_generations.parquet"
generation_model_metadata = []

for model_cfg in cfg["models"]["evaluated"]:
    print(f"\n===== PoliBench generator: {model_cfg['short_name']} | profile={PROFILE} =====")
    gpu = require_colab_gpu(float(model_cfg.get("min_gpu_vram_gb", 10)))
    print(gpu)
    model = load_model(model_cfg)
    try:
        generation_model_metadata.append(model.metadata())
        generations = run_phase4_polibench_generation(
            model=model,
            cases=cases,
            exemplars=pd.read_csv(ROOT / "data/bias_exemplars.csv"),
            output_path=generation_path,
            k=int(phase4_cfg["k_retrieval"]),
            max_cases=MAX_CASES,
            seed=seed,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=float(cfg["models"]["generation"]["temperature"]),
            checkpoint_every=checkpoint_every,
            compression=compression,
        )
        display(generations.groupby(["model", "condition"]).agg(
            rows=("case_id", "size"),
            statements=("case_id", "nunique"),
            successful=("response", lambda x: int(x.notna().sum())),
            failures=("error", lambda x: int(x.notna().sum())),
        ).reset_index())
    finally:
        model.unload()

generations = pd.read_parquet(generation_path)
expected_generators = {m["short_name"] for m in cfg["models"]["evaluated"]}
present_generators = set(generations.loc[
    generations["error"].isna() & generations["response"].notna(), "model"
].astype(str))
missing_generators = expected_generators - present_generators
if missing_generators:
    raise RuntimeError(
        f"Both generators must finish before judging. Missing successful output for: {sorted(missing_generators)}. "
        "Resolve model errors and rerun this cell; successful checkpoints are retained."
    )
print(f"Saved {len(generations)} generation records to {generation_path}")

### Quick look at generated responses

This preview shows the two conditions side-by-side for a few statement/model pairs. The full response collection is available in the expandable results section at the end of this notebook.

In [ ]:
valid_generations = generations[
    generations["error"].isna() & generations["response"].notna()
].copy()
preview = valid_generations.pivot_table(
    index=["case_id", "statement", "model"],
    columns="condition",
    values="response",
    aggfunc="first",
).reset_index()
display(preview.head(6))

## 5. Blinded pairwise judging

The same Qwen2.5-14B judge scores each response on balance, viewpoint coverage, evidence caution, false-equivalence avoidance, transparency (1–5), and favoritism (−2 to +2). For pilot/paper runs, each pair is judged twice with X/Y order reversed. Failed JSON parsing is retried; unresolved failures remain missing and are not converted to zero scores.

In [ ]:
judge_cfg = cfg["models"]["judge"]
print("Loading judge:", judge_cfg["id"])
gpu = require_colab_gpu(float(judge_cfg.get("min_gpu_vram_gb", 14)))
print(gpu)
judge = load_model(judge_cfg)
try:
    judge_model_metadata = judge.metadata()
    judgments_path = RAW_DIR / "phase4_polibench_judgments.parquet"
    judged = run_phase4_polibench_judging(
        judge_model=judge,
        generations=generations,
        output_path=judgments_path,
        judge_repeats=int(phase4_cfg["judge_repeats"]),
        max_parse_retries=int(cfg["phase4"]["judge_max_parse_retries"]),
        seed=seed,
        checkpoint_every=checkpoint_every,
        compression=compression,
    )
finally:
    judge.unload()

print(f"Saved {len(judged)} judge records to {judgments_path}")
display(judged.groupby(["generator_model"]).agg(
    attempted=("parse_success", "size"),
    parse_success_rate=("parse_success", lambda x: float(x.fillna(False).mean())),
).reset_index())

## 6. Analyze and display the results

The notebook writes alternate-specific CSV and Parquet files. It does not touch `phase4_judge_summary.csv`, `phase4_judge_quality.csv`, or the original Phase 4 generation/judgment files.

In [ ]:
summary, quality = analyze_phase4_polibench(
    judged,
    derived_dir=DERIVED_DIR,
    bootstrap_iterations=int(cfg["phase4"]["bootstrap_iterations"]),
    seed=seed,
)

if summary.empty:
    print("No parse-valid judge comparisons were available for aggregate scoring.")
else:
    print("Paired mitigation summary (95% case-bootstrap intervals)")
    display(summary.sort_values(["generator_model", "metric"]).reset_index(drop=True))

if quality.empty:
    print("No judge quality diagnostics were available.")
else:
    print("Judge validity diagnostics")
    display(quality)

    parse_threshold = float(cfg["phase4"]["capability_gate"]["minimum_parse_success_rate"])
    order_threshold = float(cfg["phase4"]["capability_gate"]["minimum_order_consistency"])
    quality_display = quality.copy()
    quality_display["parse_gate"] = np.where(
        quality_display["parse_success_rate"] >= parse_threshold, "PASS", "FAIL"
    )
    quality_display["order_gate"] = np.where(
        quality_display["order_consistency"].isna(),
        "NOT ESTIMABLE (need 2+ repeats)",
        np.where(quality_display["order_consistency"] >= order_threshold, "PASS", "FAIL"),
    )
    display(quality_display)

### Effect-size plots

Positive values mean improvement in the adaptive Reasonable Pluralism condition, except the preference-rate metric, where the value is the adaptive pairwise preference rate.

In [ ]:
if summary.empty or "metric" not in summary.columns:
    print("No valid judge scores to plot.")
else:
    for metric in ["pluralism_composite", "absolute_favoritism_reduction", "adaptive_pairwise_preference_rate"]:
        plot_df = summary[summary["metric"] == metric].copy()
        if plot_df.empty:
            print(f"No valid observations to plot for {metric}.")
            continue
        x = np.arange(len(plot_df))
        means = plot_df["mean_improvement"].to_numpy(dtype=float)
        low = plot_df["ci_low"].to_numpy(dtype=float)
        high = plot_df["ci_high"].to_numpy(dtype=float)
        yerr = np.vstack([np.maximum(0, means - low), np.maximum(0, high - means)])
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.errorbar(x, means, yerr=yerr, fmt="o", capsize=4)
        ax.axhline(0, linewidth=1)
        ax.set_xticks(x, labels=plot_df["generator_model"].astype(str).tolist(), rotation=12)
        ax.set_ylabel("Mean improvement / rate")
        ax.set_title(metric.replace("_", " ").title() + " (95% case-bootstrap CI)")
        fig.tight_layout()
        plt.show()
        fig.savefig(DERIVED_DIR / f"phase4_polibench_{metric}.png", dpi=int(cfg["analysis"]["save_png_dpi"]), bbox_inches="tight")
        plt.close(fig)

### Statement-level score table

This table averages the reversed-order judge repeats within each statement before display. It is useful for spotting where the adaptive response improved or worsened; the model-level bootstrap summary above remains the primary aggregate analysis.

In [ ]:
dimensions = [
    "balance", "viewpoint_coverage", "evidence_caution",
    "false_equivalence_avoidance", "transparency",
]
valid_judged = judged[judged["parse_success"].fillna(False)].copy()
case_rows = []
for _, r in valid_judged.iterrows():
    rec = {
        "generator_model": r["generator_model"],
        "case_id": int(r["case_id"]),
        "repeat_idx": int(r["repeat_idx"]),
    }
    for dim in dimensions + ["favoritism"]:
        sx = "x" if r["condition_x"] == "baseline" else "y"
        sy = "x" if r["condition_x"] == "adaptive_pluralism" else "y"
        rec[f"baseline_{dim}"] = float(r.get(f"{sx}_{dim}", np.nan))
        rec[f"adaptive_{dim}"] = float(r.get(f"{sy}_{dim}", np.nan))
    pref = r["overall_preference"]
    if pref == "TIE":
        rec["adaptive_preferred"] = 0.5
    else:
        winner_condition = r["condition_x"] if pref == "X" else r["condition_y"]
        rec["adaptive_preferred"] = 1.0 if winner_condition == "adaptive_pluralism" else 0.0
    case_rows.append(rec)

if case_rows:
    case_obs = pd.DataFrame(case_rows)
    for condition in ["baseline", "adaptive"]:
        cols = [f"{condition}_{d}" for d in dimensions]
        case_obs[f"{condition}_composite"] = case_obs[cols].mean(axis=1)
    case_level = case_obs.groupby(["generator_model", "case_id"], as_index=False).mean(numeric_only=True)
    statement_map = cases.set_index("case_id")["statement"].to_dict()
    case_level["statement"] = case_level["case_id"].map(statement_map)
    case_level["composite_delta"] = case_level["adaptive_composite"] - case_level["baseline_composite"]
    case_level["absolute_favoritism_reduction"] = (
        case_level["baseline_favoritism"].abs() - case_level["adaptive_favoritism"].abs()
    )
    case_level.to_csv(DERIVED_DIR / "phase4_polibench_statement_level.csv", index=False)
    display(case_level[[
        "case_id", "statement", "generator_model", "baseline_composite",
        "adaptive_composite", "composite_delta", "baseline_favoritism",
        "adaptive_favoritism", "absolute_favoritism_reduction", "adaptive_preferred",
    ]].sort_values(["generator_model", "case_id"]).reset_index(drop=True))
else:
    case_level = pd.DataFrame()
    print("No parse-valid judge rows are available for statement-level scoring.")

## 7. Expandable full responses for every statement

Every successful baseline/adaptive response is shown below in a collapsed section, grouped by statement and generator. Expand any item to inspect the complete response text, including its FOR, AGAINST, and transparency-note sections.

In [ ]:
if DISPLAY_FULL_RESPONSES:
    display_df = valid_generations.pivot_table(
        index=["case_id", "statement", "model"],
        columns="condition",
        values="response",
        aggfunc="first",
    ).reset_index()
    html_parts = [
        "<style>.pb-output{white-space:pre-wrap;overflow-wrap:anywhere;font-size:0.92em;"
        "padding:0.7em;border:1px solid #ddd;border-radius:6px;margin:0.4em 0 1em 0;}"
        ".pb-wrap details{margin:0.45em 0;padding:0.45em;border-bottom:1px solid #ddd;}"
        ".pb-wrap summary{cursor:pointer;font-weight:600;}</style><div class='pb-wrap'>"
    ]
    for _, r in display_df.sort_values(["case_id", "model"]).iterrows():
        statement_text = html.escape(str(r["statement"]))
        model_text = html.escape(str(r["model"]))
        baseline = r.get("baseline", None)
        adaptive = r.get("adaptive_pluralism", None)
        baseline = "No successful baseline output" if baseline is None or pd.isna(baseline) else str(baseline)
        adaptive = "No successful adaptive output" if adaptive is None or pd.isna(adaptive) else str(adaptive)
        html_parts.append(
            f"<details><summary>Statement {int(r['case_id'])} · {model_text} — {statement_text[:180]}</summary>"
            f"<p><b>Statement:</b> {statement_text}</p>"
            f"<h4>Baseline</h4><div class='pb-output'>{html.escape(baseline)}</div>"
            f"<h4>Adaptive Reasonable Pluralism</h4><div class='pb-output'>{html.escape(adaptive)}</div>"
            "</details>"
        )
    html_parts.append("</div>")
    display(HTML("".join(html_parts)))
else:
    print("Full response display is disabled. Set DISPLAY_FULL_RESPONSES = True and rerun this cell.")

## 8. Save manifest and prepare blinded human evaluation

The automated judge is secondary evidence. The notebook also creates a blinded human-rating sheet and a separate key file. For publication-quality claims, collect ratings from at least two independent raters and analyze reliability.

In [ ]:
human_sheet_path = DERIVED_DIR / "phase4_polibench_human_rating_sheet.csv"
human_key_path = DERIVED_DIR / "phase4_polibench_human_blind_key.parquet"
human_sheet, human_key = build_polibench_human_sheet(
    generations,
    sheet_path=human_sheet_path,
    key_path=human_key_path,
    seed=seed,
)

manifest = {
    "experiment": "alternate_phase4_polibench_statements",
    "profile": PROFILE,
    "git_sha": GIT_SHA,
    "dataset_path": str(DATA_PATH),
    "dataset_sha256": file_sha256(DATA_PATH),
    "dataset_rows": int(len(statements)),
    "profile_statement_limit": MAX_CASES,
    "prompt_version": STATEMENT_PROMPT_VERSION,
    "task_prompt": "strongest defensible case FOR and AGAINST each statement; transparency note; no forced false equivalence",
    "max_new_tokens": MAX_NEW_TOKENS,
    "temperature": float(cfg["models"]["generation"]["temperature"]),
    "retrieval_k": int(phase4_cfg["k_retrieval"]),
    "judge_repeats": int(phase4_cfg["judge_repeats"]),
    "bootstrap_iterations": int(cfg["phase4"]["bootstrap_iterations"]),
    "generator_models": generation_model_metadata,
    "judge_model": judge_model_metadata,
    "outputs": {
        "generations": str(generation_path),
        "judgments": str(judgments_path),
        "summary": str(DERIVED_DIR / "phase4_polibench_judge_summary.csv"),
        "judge_quality": str(DERIVED_DIR / "phase4_polibench_judge_quality.csv"),
        "statement_level": str(DERIVED_DIR / "phase4_polibench_statement_level.csv"),
        "human_sheet": str(human_sheet_path),
        "human_key": str(human_key_path),
    },
}
manifest_path = MANIFEST_DIR / "phase4_polibench_manifest.json"
write_json(manifest, manifest_path)
print("Manifest:", manifest_path)
print("Human-rating sheet:", human_sheet_path)
print("Blind key (keep private until ratings are complete):", human_key_path)
print("All alternate Phase 4 artifacts are stored in the profile/commit-specific Drive run.")
display(pd.DataFrame([{
    "profile": PROFILE,
    "statements_in_dataset": len(statements),
    "generation_records": len(generations),
    "judge_records": len(judged),
    "valid_judge_records": int(judged["parse_success"].fillna(False).sum()) if "parse_success" in judged else 0,
    "human_pairs": len(human_sheet),
}]))

## Interpretation guardrails

- This dataset is a statement-level stress test, not a gold-labeled bias benchmark. There is no ground-truth label saying which side is correct.
- The judge scores response quality under a declared rubric; it does not establish political neutrality or factual correctness by itself.
- A higher balance score does not mean the two sides deserve equal weight. The rubric explicitly penalizes false equivalence and overclaiming.
- The transparency note is required in both conditions for task comparability; the adaptive condition additionally uses the existing Reasonable Pluralism system prompt and retrieved bias-correction examples.
- Inspect the parse/order-consistency gates and the human ratings before making substantive claims. Smoke runs with one judge repeat cannot estimate order consistency.
- Original Phase 4 artifacts remain under their original names; this alternate run writes only `phase4_polibench_*` files.